Read races.csv file from dropzone, format column names, add metadata and load into datalake/raw schema

In [0]:
from pyspark.sql.types import StringType, IntegerType, DoubleType, StructType, StructField
from pyspark.sql.functions import col, current_timestamp, concat, lit, to_timestamp

In [0]:
races_schema = StructType([StructField("raceid", IntegerType(), False),
                              StructField("year", IntegerType(), True),
                              StructField("round", IntegerType(), True),
                              StructField("circuitid", IntegerType(), True), 
                              StructField("name", StringType(), True),
                              StructField("date", StringType(), True),
                              StructField("time", StringType(), True),
                              StructField("url", StringType(), True)
])

In [0]:
raw_df = spark.read \
    .options(header=True) \
    .options(schema=races_schema) \
    .csv("abfss://dropzone@databrickssandbox202307.dfs.core.windows.net/formula1/races.csv")

In [0]:
formatted_races_df = raw_df \
    .withColumn("date_ingested", current_timestamp()) 

In [0]:
formatted_races_df.write.mode('overwrite').partitionBy('year').parquet("abfss://datalakehouse@databrickssandbox202307.dfs.core.windows.net/schemas/formula1/raw/races")